- preface_text.json : preface text for each scenario
- categorized_lists.json : list of semantically related words (n3, n5, n7, n10)
- random_lists.json : list of arbitrary words (n3, n5, n7, n10)
- intervening_texts.json : key for each scenario. Then key for different length (1,2,3,4,5)

In [1]:
import sys
import os

# Adjust the path to where the src folder is located
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

In [23]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from language_models.dictionary_corpus import Dictionary
from collections import defaultdict
import torch.nn as nn
from torch.nn.functional import scaled_dot_product_attention
import language_models.model as m
import math
import torch.nn.functional as F


In [17]:
device = torch.device('cpu')

## 1. verify if words from lists are in vocab

In [3]:
# File paths
vocab_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data/vocab.txt"
nouns_path = "/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/nouns_categorized.txt"

# Load vocabulary
with open(vocab_path, "r", encoding="utf-8") as f:
    vocab = set(word.strip() for word in f)

# Load noun words
with open(nouns_path, "r", encoding="utf-8") as f:
    nouns = [word.strip().lower() for word in f if word.strip()]

# Find missing words
missing_words = [word for word in nouns if word not in vocab]

# Print or save
print(f"{len(missing_words)} words not found in vocab:")
for word in missing_words:
    print(word)


389 words not found in vocab:
dormer
alcove
livingroom
rowboat
motorboat
cargoship
clippership
outrigger
speedboat
runabout
hydrofoil
skiff
catamaran
canary
parakeet
blackbird
wren
oriole
starling
titmouse
bluejay
finch
buzzard
flamingo
lark
tangelo
tangerine
blackberry
prunes
cantaloupe
blueberry
boysenberry
raisin
nectarine
kumquat
papaya
honeydew
pliers
sandpaper
awl
sander
plumb
vise
nailset
crowbar
hatchet
ripsaw
drillpress
hacksaw
rasp
jigsaw
sleet
chinook
duststorm
drizzle
breezy
cloudburst
krypton
rubidium
cesium
john
jim
tom
joe
dick
mike
george
harry
steve
larry
paul
sam
dave
fred
charles
jerry
pete
rick
elmer
bruce
gary
karl
henry
ron
jeff
ralph
entomology
bacteriology
biophysics
embryology
agronomy
chicago
helsinki
toronto
london
paris
miami
boston
philadelphia
detroit
dallas
rome
bombay
cleveland
pittsburgh
denver
montreal
tokyo
moscow
atlanta
berlin
stockholm
jerusalem
cairo
vancouver
reno
madrid
memphis
shanghai
copenhagen
naples
brussels
venice
kazoo
xylophone
celesta
b

Those words have already been filtered so they're not in the json

In [4]:
import json

# File paths
json_path = "/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/random_lists.json"
vocab_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data/vocab.txt"

# Load vocab as a lowercase set
with open(vocab_path, "r", encoding="utf-8") as f:
    vocab = set(word.strip() for word in f)

# Load JSON data
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

missing_words = set()
for key, lists in data.items():
    for word_list in lists:
        for word in word_list:
            if word.lower() not in vocab:
                missing_words.add(word.lower())

print(f"{len(missing_words)} lowercase words not in vocab:")
for word in sorted(missing_words):
    print(word)

# Filter lists
filtered_data = {}
for key, lists in data.items():
    filtered_data[key] = [
        word_list for word_list in lists
        if all(word.lower() in vocab for word in word_list)
    ]

# (Optional) Save cleaned data back to file
# cleaned_path = "/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/categorized_lists_cleaned.json"
# with open(cleaned_path, "w", encoding="utf-8") as f:
#     json.dump(filtered_data, f, indent=2)

# print(f"Cleaned data saved to: {cleaned_path}")


0 lowercase words not in vocab:


# Now create test set

In [73]:
class WMTestDataset(Dataset):
    def __init__(self, sentences_path, markers_path, dictionary):
        """
        sentences_path: path to the sentence file (one sentence per line)
        markers_path: path to marker file (CSV/TSV with columns: markers, stimid, list_len, prompt_len)
                      markers column contains array-like strings (e.g. '[0,0,1,1,...]')
        dictionary: object with attribute 'word2idx' (dict mapping words to indices), and
                    should contain an "<unk>" key for unknown words.
        """
        self.dictionary = dictionary
        
        # Load sentences
        with open(sentences_path, "r", encoding="utf-8") as f:
            self.sentences = [line.strip().split() for line in f if line.strip()]
        
        # Load markers metadata file with pandas
        self.markers_df = pd.read_csv(markers_path, sep="\t", converters={
            'markers': lambda x: list(map(int, x.strip('[]').split(',')))
        })
        
        # Sanity check: same number of sentences and markers rows
        assert len(self.sentences) == len(self.markers_df), \
            f"Mismatch: {len(self.sentences)} sentences vs {len(self.markers_df)} marker entries"
    
    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        row = self.markers_df.iloc[idx]
        
        markers = row['markers']
        stimid = row['stimid']
        list_len = row['list_len']
        prompt_len = row['prompt_len']
        
        encoded_sentence = [
            self.dictionary.word2idx.get(word, self.dictionary.word2idx.get("<unk>"))
            for word in sentence
        ]
        
        list1 = [sentence[i] for i, tag in enumerate(markers) if tag == 1]
        list2 = [sentence[i] for i, tag in enumerate(markers) if tag == 3]
        
        list1_encoded = [
        encoded_sentence[i] for i, tag in enumerate(markers) if tag == 1
        
        ]
        list2_encoded = [
            encoded_sentence[i] for i, tag in enumerate(markers) if tag == 3
        ]
        
        condition = (list_len, prompt_len)
        
        sample = {
            "sentence": sentence,              # list of tokens (words)
            "encoded_sentence": encoded_sentence, 
            'list1_encoded':list1_encoded,
            "list1":list1,
            "list2_encoded": list2_encoded,
            "list2":list2,
            "stimid": stimid,  
            "condition" : condition,
            "marker":markers
            # "list_len": list_len,              # int
            # "prompt_len": prompt_len           # int
        }
        return sample


In [74]:
sentence_path = '/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/rnn_input_files/categorized_lists_sce1_control.txt'
marker_path = '/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/rnn_input_files/categorized_lists_sce1_control_markers.txt'
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"
dictionary = Dictionary(data_path)
wm_dataset = WMTestDataset(sentence_path, marker_path, dictionary)

In [75]:
def create_cond_dataloader(dataset):
    """"
    For each conditions (same list len and intervening text len) sentences are the same length, so we can batch by condition. 
    To do so we do one dataloader per conditon. The max batch size is 230 (1 batch per condition)
    """
    condition_loaders = {}
    for (list_len, prompt_len) in dataset.conditions:  # set of tuples
        condition_dataset = [
            sample for sample in dataset
            if sample["list_len"] == list_len and sample["prompt_len"] == prompt_len
        ]
        loader = DataLoader(condition_dataset, batch_size=230, shuffle=False)
        condition_loaders[(list_len, prompt_len)] = loader
    
    return condition_loaders

In [81]:
def collate_fn(batch):
    """Custom collate function to properly handle sentences as lists of strings."""
    sentences = [item["sentence"] for item in batch] 
    list1 = [item["list1"] for item in batch] 
    list2 = [item["list2"] for item in batch] 
    encoded_sentences = torch.stack([torch.tensor(item["encoded_sentence"]) for item in batch])  
    list1_encoded = torch.stack([torch.tensor(item["list1_encoded"]) for item in batch])
    list2_encoded = torch.stack([torch.tensor(item["list2_encoded"]) for item in batch])
    marker = torch.stack([torch.tensor(item['marker']) for item in batch])
    conditions = [item["condition"] for item in batch]

    return {
        "sentence": sentences,
        "encoded_sentence": encoded_sentences,
        "list1": list1,
        "list2": list2,
        "list1_encoded":list1_encoded,
        "list2_encoded":list2_encoded,
        "condition": conditions,
        "marker": marker
    }


In [82]:
"""
A simpler approach is to use a normal dataloader with a batch size of 230 (or any n divisible by 230)
"""

dataloader = DataLoader(wm_dataset, batch_size=230, collate_fn=collate_fn)


# Load model

In [83]:
class CBR_RNN(nn.Module):
    # goal here is to reuse CBR_RNN but with scaled dot product attention for more efficient computations.
    # Also I got rid of options such as loading pretrained embeddings, and ablating attention to simplify the code.
    # In the future if those options are needed, they can still be copy pasted from William's code as the structure hasn't changed
    def __init__(self, ntoken, ninp, nhid, nheads, dropout=0.5, device=None):
        super().__init__()
        # same layers as Timkey
        self.device = device
        self.nheads = nheads
        self.tanh = nn.Tanh()
        self.drop = nn.Dropout(dropout)
        self.score_attn = nn.Softmax(dim=-1)
        self.encoder = nn.Embedding(ntoken, ninp)
        self.q = nn.Linear(ninp + nhid, nhid)
        self.intermediate_h = nn.Linear(nhid * 4, nhid * 4)
        self.decoder = nn.Linear(nhid, ntoken)
        self.q_norm = torch.nn.LayerNorm(nhid)
        self.int_norm = torch.nn.LayerNorm(nhid * 4)
        self.f_norm = torch.nn.LayerNorm(nhid * 3)
        self.nhid = nhid
        self.final_h = nn.Linear(nhid * 4, nhid * 3)
        self.multihead_attn = nn.MultiheadAttention(
            embed_dim=nhid, num_heads=nheads, batch_first=True
        )

        self.init_weights()

    def init_weights(self):
        """Initialize model weights for better training dynamics"""
        # General initialization
        for name, param in self.named_parameters():
            if "weight" in name:
                if "norm" in name:
                    nn.init.ones_(param)
                elif "encoder" in name:
                    nn.init.normal_(param, mean=0, std=0.01)
                elif "decoder" in name:
                    nn.init.normal_(param, mean=0, std=0.01)
                else:
                    # Standard He initialization for processing layers
                    nn.init.kaiming_normal_(param, mode="fan_in", nonlinearity="tanh")
            elif "bias" in name:
                nn.init.zeros_(param)

    def init_cache(self, observation, nheads):
        """Initialize hidden state and attention caches with better initialization strategy"""
        if len(observation.size()) > 1:
            bsz = observation.size(dim=-1)
        else:
            bsz = 1

        hidden = torch.zeros(1, bsz, self.nhid).to(self.device) 
        if nheads == 1:
            key_cache = torch.zeros(bsz, 1, 1, self.nhid).to(self.device) 
            value_cache = torch.zeros(bsz, 1, 1, self.nhid).to(self.device) 
        else:
            key_cache = torch.zeros(bsz, 1, self.nhid).to(self.device) 
            value_cache = torch.zeros(bsz, 1, self.nhid).to(self.device) 
        return hidden, key_cache, value_cache


    def update_cache(self, key_cache, value_cache, hidden, key_cache_i, value_cache_i, hidden_i, nheads):
        hidden_i = hidden_i.unsqueeze(0)
        hidden = torch.cat((hidden, hidden_i), dim=0)
        if nheads == 1:
                key_cache_i = key_cache_i.unsqueeze(1).unsqueeze(1)
                value_cache_i = value_cache_i.unsqueeze(1).unsqueeze(1)
                key_cache = torch.cat((key_cache, key_cache_i), dim=2)
                value_cache = torch.cat((value_cache, value_cache_i), dim=2)
        else:
            key_cache_i = key_cache_i.unsqueeze(1)
            value_cache_i = value_cache_i.unsqueeze(1)
            key_cache = torch.cat((key_cache, key_cache_i), dim=1)
            value_cache = torch.cat((value_cache, value_cache_i), dim=1)
            
        return key_cache, value_cache, hidden
    
    @staticmethod
    def temperature_attention(query, key, value, temperature, gumbel_softmax=None, attn_mask=None,dropout_p=0.0,
        is_causal=False, scale=None, enable_gqa=False) -> torch.Tensor:
        L, S = query.size(-2), key.size(-2)
        scale_factor = 1 / math.sqrt(query.size(-1)) if scale is None else scale
        attn_bias = torch.zeros(L, S, dtype=query.dtype, device=query.device)
        if is_causal:
            assert attn_mask is None
            temp_mask = torch.ones(L, S, dtype=torch.bool).tril(diagonal=0)
            attn_bias.masked_fill_(temp_mask.logical_not(), float("-inf"))
            attn_bias.to(query.dtype)

        if attn_mask is not None:
            if attn_mask.dtype == torch.bool:
                attn_bias.masked_fill_(attn_mask.logical_not(), float("-inf"))
            else:
                attn_bias = attn_mask + attn_bias

        if enable_gqa:
            key = key.repeat_interleave(query.size(-3)//key.size(-3), -3)
            value = value.repeat_interleave(query.size(-3)//value.size(-3), -3)

        attn_weight = query @ key.transpose(-2, -1) * scale_factor
        attn_weight += attn_bias
        if gumbel_softmax: 
            attn_weight = torch.gumbel_softmax(attn_weight, tau=temperature, hard=False, dim=-1)
        else : 
            attn_weight = attn_weight/temperature
            attn_weight = torch.softmax(attn_weight, dim=-1)
        attn_weight = torch.dropout(attn_weight, dropout_p, train=True)
        return attn_weight @ value
    
    def attention_layer(self, query, key_cache, value_cache, nheads, temperature, gumbel_softmax):
        if nheads == 1:
                query = query.unsqueeze(1)
                
                # Ensure all tensors are on the same device
                if query.device != key_cache.device:
                    key_cache = key_cache.to(query.device)
                if query.device != value_cache.device:
                    value_cache = value_cache.to(query.device)
                if temperature==1:  
                    try:
                        attn_output = scaled_dot_product_attention(
                            query, key_cache, value_cache, is_causal=False
                        )
                    except Exception as e:
                        logging.error(f"Error in attention computation: {str(e)}")
                        raise
                    attn = attn_output.squeeze(1).squeeze(1)
                    del attn_output  # No longer needed after squeezing
                    query = query.squeeze(1).squeeze(1)
                else:
                    try:
                        attn_output = self.temperature_attention(
                            query, key_cache, value_cache, temperature, is_causal=False
                        )
                    except Exception as e:
                        logging.error(f"Error in attention computation: {str(e)}")
                        raise
                    attn = attn_output.squeeze(1).squeeze(1)
                    del attn_output  # No longer needed after squeezing
                    query = query.squeeze(1).squeeze(1)

            
        else:
            attn_output, _ = self.multihead_attn(
                query, key_cache, value_cache, is_causal=False
            )
            attn = attn_output.squeeze(1)
            del attn_output  # No longer needed after squeezing
            query = query.squeeze(1)
            
        return attn, query
    
    def intermediate_layers(self, i, emb, query, attn, hidden):
        intermediate_input = torch.cat((emb[i], query, attn, hidden[-1]), -1)
        del query, attn  
        intermediate = self.drop(
            self.tanh(self.int_norm(self.intermediate_h(intermediate_input)))
        )
        del intermediate_input  
        final_output = self.drop(self.tanh(self.f_norm(self.final_h(intermediate))))
        del intermediate  
        key_cache_i, value_cache_i, hidden_i = final_output.split(self.nhid, dim=-1)
        del final_output
        return key_cache_i, value_cache_i, hidden_i
    
    def get_query(self, emb, hidden):
        combined = torch.cat((emb, hidden[-1]), -1)
        query = self.drop(self.tanh(self.q_norm(self.q(combined))))
        del combined  # No longer needed after creating query
        query = query.unsqueeze(1)
        return query
    
    def forward(self, observation, initial_cache, nheads, temperature, gumbel_softmax):
        seq_len = observation.size(0)
        hidden, key_cache, value_cache = initial_cache

        # 1. Encode observations
        emb = self.drop(self.encoder(observation))
        del observation  # No longer needed after encoding
        
        for i in range(seq_len):
            # 2. Concatenate with previous hidden state

            
            query = self.get_query(emb[i], hidden)
            
            attn, query = self.attention_layer(query, key_cache, value_cache, nheads, temperature, gumbel_softmax)

            key_cache_i, value_cache_i, hidden_i = self.intermediate_layers(i, emb, query, attn, hidden)
            
            key_cache, value_cache, hidden = self.update_cache(key_cache, value_cache, hidden, key_cache_i, value_cache_i, hidden_i, nheads)
            
            del key_cache_i, value_cache_i, hidden_i  # No longer needed after concatenation

        decoded = self.decoder(hidden[1:])

        return decoded, hidden

In [84]:
model = CBR_RNN(50001, 128, 128, 1, 0, device)
check = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/cbr_1h_128_gumbel_softmax/epoch_40.pt', map_location='cpu')
model.load_state_dict(check['model_state_dict'])

<All keys matched successfully>

Eux c'est mot par mot : NLL(list2)/NLL(list1)*100

In [ ]:
all_repeat_surprisals = []
model.eval()
# Forward pass with hidden state update word by word
with torch.no_grad():
    for batch in dataloader:
        out = None
        list1 = batch["list1"]
        encoded_sentence = batch["encoded_sentence"]
        condition = batch["condition"]
        marker = batch["marker"]
        batch_size, seq_len = encoded_sentence.shape
        input_seq = encoded_sentence[:, :-1].transpose(0, 1)  # (seq_len-1, batch_size)
        target_seq = encoded_sentence[:, 1:].transpose(0, 1)
        cache = model.init_cache(input_seq, 1)
        output, hidden = model(input_seq, cache, 1, 0.1, True)
        log_probs = F.log_softmax(output, dim=-1)  # shape (seq_len-1, batch_size, vocab_size)
        
        nll_loss = F.nll_loss(
                    log_probs.reshape(-1, log_probs.size(-1)),  # ( (seq_len-1)*batch_size, vocab_size )
                    target_seq.reshape(-1),                         # ((seq_len-1)*batch_size)
                    reduction='none'
                )
                
                # Reshape back to (seq_len-1, batch_size)
        nll_loss = nll_loss.view(seq_len - 1, batch_size).transpose(0, 1)  # (batch_size, seq_len-1)        
        mask_list1 = (marker[:, 1:] == 1)  # remove first token since nll_loss aligns with shifted target
        mask_list2 = (marker[:, 1:] == 3)
        
        # Extract surprisal for each list and reshape
        # Number of tokens in each list should be condition[0][0]*2 (including punctuation)
        list_len = condition[0][0] * 2
        
        surprisal_list1 = nll_loss[mask_list1].view(batch_size, list_len)
        surprisal_list2 = nll_loss[mask_list2].view(batch_size, list_len)
        
        # Select repeated word indices (odd positions assuming repeats are at odd indices)
        word_indices = torch.arange(0, condition[0][0]*2, step=2)  # e.g., 1, 3, 5, ...
        word_indices = word_indices[1:]#get rid of first word of the list
        # print(word_indices)
        # for i in word_indices:
        #     words_at_indices = list1[0][i]
        #     print(words_at_indices)
        surprisal1_repeats = surprisal_list1[:, word_indices]
        surprisal2_repeats = surprisal_list2[:, word_indices]
        
        # Compute repeat surprisal ratio as percentage
        repeat_surprisal = (surprisal2_repeats / surprisal1_repeats) * 100
        
        all_repeat_surprisals.append(repeat_surprisal)
        
        
        

In [143]:
for i in range(len(all_repeat_surprisals)):
    print(all_repeat_surprisals[i].shape)

torch.Size([230, 2])
torch.Size([230, 4])
torch.Size([230, 6])
torch.Size([230, 9])
torch.Size([230, 2])
torch.Size([230, 4])
torch.Size([230, 6])
torch.Size([230, 9])
torch.Size([230, 2])
torch.Size([230, 4])
torch.Size([230, 6])
torch.Size([230, 9])
torch.Size([230, 2])
torch.Size([230, 4])
torch.Size([230, 6])
torch.Size([230, 9])
torch.Size([230, 2])
torch.Size([230, 4])
torch.Size([230, 6])
torch.Size([230, 9])


In [114]:
#def eval(model, test_dataloader):

all_repeat_surprisals = []
model.eval()
# Forward pass with hidden state update word by word
with torch.no_grad():
    for batch in dataloader:
        out = None
        encoded_sentence = batch["encoded_sentence"]
        condition = batch["condition"]
        marker = batch["marker"]
        batch_size, seq_len = encoded_sentence.shape
        
        
        sentence_nll = torch.zeros(batch_size, seq_len)
        for i in range(1, seq_len):
            input = encoded_sentence[:, :i] 
            target = encoded_sentence[:, i] 
            cache = model.init_cache(input, 1)
            output, hidden = model(input, cache, 1, 0.1, True)

            log_probs = F.log_softmax(output, dim=-1)                             

            nll_loss = F.nll_loss(
                    log_probs[:,i-1,:],
                    target,
                    reduction='none'
                )
            sentence_nll[:, i] = nll_loss
            
        mask_list1 = (marker==1)
        mask_list2 = (marker==3)
        
        surprisal_list1 = sentence_nll[mask_list1]
        surprisal_list1 = surprisal_list1.view(batch_size, condition[0][0]*2) #include punctuation
        surprisal_list2 = sentence_nll[mask_list2]
        surprisal_list2 = surprisal_list2.view(batch_size, condition[0][0]*2)
        
        
        word_indices = torch.arange(1, condition[0][0], step=2)
        
        surprisal1_repeats = surprisal_list1[:, word_indices]
        surprisal2_repeats = surprisal_list2[:, word_indices]

        # compute repeat surprisal (batch_size, num_repeats)
        repeat_surprisal = (surprisal2_repeats / surprisal1_repeats) * 100
        all_repeat_surprisals.append(repeat_surprisal)

# Optionally stack to get shape (total_batches * batch_size, num_repeats)
all_repeat_surprisals = torch.cat(all_repeat_surprisals, dim=0)

KeyboardInterrupt: 

In [111]:
repeat_surprisal[0].shape

torch.Size([230])

In [93]:
print(sentence_nll.shape)
print(marker.shape)

torch.Size([230, 51])
torch.Size([230, 51])


# pts d'attention sur l'initialisation du cache. cahce reinitialisé a chaque batch (donc reste le meme cache sur tte la durée de la bptt) pdt l'entrainement. ici reinitialisé pr chaque longueur de bptt. 